In [ ]:
from typing import Sequence
from fealpy.backend import backend_manager as bm
from fealpy.backend import TensorLike

from fealpy.mesh import UniformMesh2d
from fealpy.functionspace.lagrange_fe_space import LagrangeFESpace
from fealpy.decorator import cartesian
from fealpy.fem import BilinearForm, ScalarDiffusionIntegrator
from fealpy.fem import LinearForm, ScalarSourceIntegrator
from fealpy.fem import DirichletBC
from fealpy.solver import cg

device = 'cpu'
bm.set_backend('pytorch')
bm.set_default_device(device)

class Exp0002():
    def __init__(self):
        self.domain = [0, 1, 0, 1] 

    @cartesian
    def solution(self, p: TensorLike) -> TensorLike:
        """Compute exact solution"""
        x = p[..., 0]
        y = p[..., 1]
        pi = bm.pi
        return bm.sin(pi * x) * bm.sin(pi * y)

    @cartesian
    def diffusion_coef(self, p: TensorLike) -> TensorLike:
        """Variable diffusion coefficient k(x, y)"""
        x = p[..., 0]
        y = p[..., 1]
        return 1.0 + x + y

    @cartesian
    def source(self, p: TensorLike) -> TensorLike:
        """f = -div(k grad u), where k = 1 + x + y"""
        x = p[..., 0]
        y = p[..., 1]
        pi = bm.pi

        u = bm.sin(pi * x) * bm.sin(pi * y)

        ux = pi * bm.cos(pi * x) * bm.sin(pi * y)
        uy = pi * bm.sin(pi * x) * bm.cos(pi * y)

        k = 1.0 + x + y

        # -div(k grad u)
        # = -kx * ux - ky * uy - k * laplace(u)
        # kx = 1, ky = 1, laplace(u) = -2*pi^2*u
        return 2 * pi**2 * k * u - ux - uy

    @cartesian
    def dirichlet(self, p: TensorLike) -> TensorLike:
        """Dirichlet boundary condition"""
        return self.solution(p)

PDE = Exp0002()

domain = PDE.domain
nx, ny = 100, 100

hx = (domain[1] - domain[0])/nx
hy = (domain[3] - domain[2])/ny

mesh = UniformMesh2d((0, nx, 0, ny), h=(hx, hy), origin=(domain[0], domain[2]), ftype=bm.float32)

cqf = mesh.quadrature_formula(4, 'cell')
bcs, ws = cqf.get_quadrature_points_and_weights()
ps = mesh.bc_to_point(bcs)

space= LagrangeFESpace(mesh, p=1)
uh = space.function()
bform = BilinearForm(space)
DI = ScalarDiffusionIntegrator(PDE.diffusion_coef(ps))
bform.add_integrator(DI)

lform = LinearForm(space)
SI = ScalarSourceIntegrator(PDE.source(ps))
lform.add_integrator(SI)

A = bform.assembly()
F = lform.assembly()

A, F = DirichletBC(space, gd=PDE.solution).apply(A, F)




c:\Users\Rainbow\.conda\envs\eit\Lib\site-packages\torch\utils\_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
c:\Users\Rainbow\.conda\envs\eit\Lib\site-packages\torch\utils\_device.py:106: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return func(*args, **kwargs)


In [6]:
A = A.to_scipy().tocsr()

In [7]:
from hybrid_gs_mionet import hybrid_gs_mionet
x, info = hybrid_gs_mionet(
    A,
    F,
    correction_period=1600,
    maxiter=100000,
    rtol=1e-10,
    check_every=100,
    device="cpu",
)


iter=0, rel_res=1.000000e+00, abs_res=2.026234e-01
iter=100, rel_res=8.533352e-01, abs_res=1.729056e-01
iter=200, rel_res=7.316732e-01, abs_res=1.482541e-01
iter=300, rel_res=6.286371e-01, abs_res=1.273766e-01
iter=400, rel_res=5.408407e-01, abs_res=1.095870e-01
iter=500, rel_res=4.657412e-01, abs_res=9.437005e-02
iter=600, rel_res=4.013329e-01, abs_res=8.131941e-02
iter=700, rel_res=3.459924e-01, abs_res=7.010613e-02
iter=800, rel_res=2.983821e-01, abs_res=6.045918e-02
iter=900, rel_res=2.573852e-01, abs_res=5.215226e-02
iter=1000, rel_res=2.220604e-01, abs_res=4.499461e-02
iter=1100, rel_res=1.916087e-01, abs_res=3.882439e-02
iter=1200, rel_res=1.653490e-01, abs_res=3.350357e-02
iter=1300, rel_res=1.426987e-01, abs_res=2.891410e-02
iter=1400, rel_res=1.231581e-01, abs_res=2.495471e-02
iter=1500, rel_res=1.062979e-01, abs_res=2.153844e-02
iter=1600, rel_res=9.174893e-02, abs_res=1.859048e-02
iter=1700, rel_res=7.919331e-02, abs_res=1.604641e-02
iter=1800, rel_res=6.835729e-02, abs_res

In [9]:
print(bm.mean(bm.abs(PDE.solution(mesh.node) - x)))


tensor(3.2268e-05, dtype=torch.float64)


C:\Users\Rainbow\AppData\Local\Temp\ipykernel_23676\3976886058.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(bm.mean(bm.abs(PDE.solution(mesh.node) - x)))
